# R&D — Acte d'Engagement : analyse et remplissage

Notebook pour analyser la structure du PDF acte d'engagement et tester le remplissage des champs.

**Workflow :**
1. Charger le PDF et inspecter les spans (détection des zones pointillées)
2. Identifier les labels et mots-clés présents dans le document
3. Tester le remplissage avec des données fictives
4. Visualiser chaque page du résultat
5. Sauvegarder le PDF rempli dans `output/`

**Prérequis :** Placer le PDF vierge de l'acte d'engagement dans `input/`

In [ ]:
import sys, os
from pathlib import Path
import fitz  # pymupdf — lecture et modification de PDFs
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import io

# ── Chemins ──────────────────────────────────────────────────────────────
# ROOT pointe sur la racine du projet pour pouvoir importer les services
ROOT    = Path("../..")
RD_DIR  = Path(".")
INPUT   = RD_DIR / "input"   # PDFs vierges à déposer ici
OUTPUT  = RD_DIR / "output"  # PDFs remplis générés ici
OUTPUT.mkdir(exist_ok=True)

# Nécessaire pour que Python trouve app/services/acte_engagement_service.py
sys.path.insert(0, str(ROOT))

# ── Chargement du PDF ────────────────────────────────────────────────────
# On prend le premier PDF trouvé dans input/
# Si plusieurs PDFs sont présents, changer pdfs[0] en pdfs[N]
pdfs = list(INPUT.glob("*.pdf"))
print("PDFs disponibles :", pdfs if pdfs else "⚠️  Aucun PDF — placer un acte d'engagement vierge dans input/")

PDF_PATH = pdfs[0] if pdfs else None

## 1. Inspection des spans — quels champs sont détectables ?

On parcourt toutes les lignes du PDF et on affiche :
- Les spans de **points** (zones à remplir) avec leur bbox
- Le **texte de la ligne** complète pour identifier le label associé

In [ ]:
_DOT_CHARS = set('._…\u2026\u00b7·\u2025\u22ef\u0085')


def span_text(span: dict) -> str:
    """Reconstruit le texte d'un span rawdict depuis ses caractères."""
    return "".join(c["c"] for c in span.get("chars", []))


def find_dot_zones(spans: list) -> list[tuple]:
    """
    Trouve toutes les séquences de points consécutifs dans une ligne (rawdict).
    Retourne : liste de (x0, y0, x1, y1, fontsize)
    """
    zones = []
    for span in spans:
        zone_chars: list = []
        for char in span.get("chars", []):
            if char["c"] in _DOT_CHARS:
                zone_chars.append(char)
            else:
                if len(zone_chars) >= 3:
                    zones.append((
                        zone_chars[0]["bbox"][0],
                        min(c["bbox"][1] for c in zone_chars),
                        zone_chars[-1]["bbox"][2],
                        max(c["bbox"][3] for c in zone_chars),
                        span.get("size", 9.5),
                    ))
                zone_chars = []
        if len(zone_chars) >= 3:
            zones.append((
                zone_chars[0]["bbox"][0],
                min(c["bbox"][1] for c in zone_chars),
                zone_chars[-1]["bbox"][2],
                max(c["bbox"][3] for c in zone_chars),
                span.get("size", 9.5),
            ))
    return zones


assert PDF_PATH, "Aucun PDF dans input/ — stoppé ici"
doc = fitz.open(str(PDF_PATH))
print(f"PDF : {PDF_PATH.name}  |  {doc.page_count} page(s)\n")

for p_idx, page in enumerate(doc):
    zones_found = False
    for block in page.get_text("rawdict").get("blocks", []):
        for line in block.get("lines", []):
            spans = line.get("spans", [])
            # rawdict : reconstruire le texte depuis les chars
            full_text = "".join(span_text(s) for s in spans)
            zones = find_dot_zones(spans)
            if zones:
                if not zones_found:
                    print(f"── Page {p_idx + 1} ──────────────────────────")
                    zones_found = True
                print(f"  Ligne : {full_text.strip()!r}")
                for i, (x0, y0, x1, y1, fs) in enumerate(zones):
                    print(f"    → zone[{i}]  x0={x0:.1f}  y1={y1:.1f}  largeur={x1-x0:.1f}pt  taille={fs:.1f}")

print("\n✅ Inspection terminée")

## 2. Visualisation — surligner les zones détectées

Affiche chaque page avec les zones de points surlignées en rouge pour confirmer la détection visuelle.

In [ ]:
def render_page_with_zones(page, dpi=120):
    """Rendu d'une page avec les zones de points surlignées en rouge (rawdict)."""
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

    scale = dpi / 72
    fig, ax = plt.subplots(1, 1, figsize=(8, 11))
    ax.imshow(img)
    ax.axis("off")

    for block in page.get_text("rawdict").get("blocks", []):
        for line in block.get("lines", []):
            for (x0, y0, x1, y1, _) in find_dot_zones(line.get("spans", [])):
                rect = patches.Rectangle(
                    (x0 * scale, y0 * scale),
                    (x1 - x0) * scale, (y1 - y0) * scale,
                    linewidth=1.5, edgecolor='red', facecolor='red', alpha=0.3
                )
                ax.add_patch(rect)
    return fig


for p_idx, page in enumerate(doc):
    fig = render_page_with_zones(page)
    fig.suptitle(f"Page {p_idx + 1} — zones détectées (rouge)", fontsize=10)
    plt.tight_layout()
    plt.show()

## 3. Remplissage avec données de test

Appel au service `fill_acte_engagement` avec des données fictives.  
Adapter `TYPE` selon le document : `"physique"` | `"morale"` | `"groupement"`

In [ ]:
from app.services.acte_engagement_service import fill_acte_engagement

# ── Type de soumissionnaire ───────────────────────────────────────────────
TYPE = "physique"   # "physique" | "morale" | "groupement"

# ── Données de test complètes ─────────────────────────────────────────────
DATA = dict(
    type_soumissionnaire = TYPE,

    # ── Commun à tous les types ──────────────────────────────────────────
    signataire_nom   = "Mohammed Benali, Directeur Général",
    adresse_domicile = "12 Rue Hassan II, Casablanca",
    telephone        = "0522 123 456",
    fax              = "0522 789 012",
    email            = "contact@abi-consulting.ma",
    rib              = "007 780 0000012345678901 23",   # 24 positions
    cnss             = "1234567",
    rc_localite      = "Casablanca",
    rc_numero        = "RC 145853",
    taxe_pro         = "TP 56789012",
    ice              = "002579010000023",

    # ── Personne morale (ignoré si TYPE != "morale") ─────────────────────
    raison_sociale   = "ABI Consulting",
    forme_juridique  = "SARL AU",
    capital_social   = "100.000 MAD",
    adresse_siege    = "Imm 30, Appt 08, Rue Moulay Ahmed Loukili, Rabat",

    # ── Groupement (ignoré si TYPE != "groupement") ──────────────────────
    membres_groupement = "Société A\nSociété B",

    # ── Lieu et date ─────────────────────────────────────────────────────
    fait_a_lieu = "Casablanca",
    fait_a_date = "16/03/2026",

    signature_bytes = None,
    cachet_bytes    = None,
)

# Chargement automatique signature + cachet depuis rd/signature/assets/
ASSETS = ROOT / "rd" / "signature" / "assets"
for key, fname in [("signature_bytes", "signature.jpg"), ("cachet_bytes", "cachet.png")]:
    p = ASSETS / fname
    if p.exists():
        DATA[key] = p.read_bytes()
        print(f"✅ {fname} chargé")

pdf_bytes    = PDF_PATH.read_bytes()
result_bytes = fill_acte_engagement(pdf_bytes, **DATA)
print(f"\n✅ PDF rempli — {len(result_bytes):,} octets")

## 4. Visualisation du résultat

Comparaison avant / après pour chaque page.

In [ ]:
def page_to_img(page, dpi=120):
    """
    Convertit une fitz.Page en image PIL RGB.

    Paramètres
    ----------
    page : fitz.Page — page pymupdf
    dpi  : int       — résolution de rendu

    Retourne
    --------
    PIL.Image.Image
    """
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


# Ouvrir le PDF rempli depuis les bytes en mémoire
doc_result = fitz.open(stream=result_bytes, filetype="pdf")

# Affichage côte à côte avant / après pour chaque page
# → permet de vérifier visuellement que chaque champ est bien rempli
#   et que les points ont été correctement effacés
for p_idx in range(doc.page_count):
    fig, axes = plt.subplots(1, 2, figsize=(16, 11))

    axes[0].imshow(page_to_img(doc[p_idx]))        # PDF original (vierge)
    axes[0].set_title(f"AVANT — Page {p_idx + 1}", fontsize=10)
    axes[0].axis("off")

    axes[1].imshow(page_to_img(doc_result[p_idx])) # PDF après remplissage
    axes[1].set_title(f"APRÈS — Page {p_idx + 1}", fontsize=10)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

## 5. Sauvegarde

In [ ]:
# Nom de fichier dynamique selon le type choisi
# ex: acte_engagement_rempli_morale.pdf
out_path = OUTPUT / f"acte_engagement_rempli_{TYPE}.pdf"
out_path.write_bytes(result_bytes)
print(f"✅ Sauvegardé → {out_path}")